# Train models

In [1]:
import sys
working_directory = "/home/icb/kemal.inecik/work/codes/sctram"
sys.path.append(working_directory)

import numpy as np
import os
import itertools
import subprocess
import time

In [2]:
dataset_dir = "/home/icb/kemal.inecik/lustre_workspace/temp_sctram_data"
helpers_directory = os.path.join(os.getcwd(), "helper")
logs_directory = os.path.join(os.getcwd(), "logs")

In [3]:
def is_job_running(job_name):
    result = subprocess.run(['squeue', '-o', '%.28i %.28j %.28u %R', '-u', 'kemal.inecik'],  capture_output=True, text=True)
    jobs = [[j.strip() for j in i.split()]for i in result.stdout.strip().split('\n')]
    for jobid, jobname, jobuser, jobnode in jobs:
        if job_name == jobname:
            return True
    return False

```
#SBATCH -J {job_name}
#SBATCH -p gpu_p
#SBATCH --qos=gpu_normal
#SBATCH --gres=gpu:1
#SBATCH -c 6
#SBATCH --mem=159G
#SBATCH --nice=0
#SBATCH -t 23:50:00
#SBATCH -o {log_file}
#SBATCH -e {log_file}
```

```
#SBATCH -J {job_name}
#SBATCH -p cpu_p
#SBATCH --qos cpu_normal
#SBATCH -c 32
#SBATCH --mem=159G
#SBATCH --nice=0
#SBATCH -t 1-23:50:00
#SBATCH -o {log_file}
#SBATCH -e {log_file}
```

In [4]:
step_per_epoch = 841922 * 0.25 * 0.8 / 512
print(step_per_epoch)

epochs = list(itertools.chain(range(20, 52), range(52, 100, 2), range(100, 200, 10), range(200, 400, 20)))
epochs = list(np.arange(0, 20, 0.10)) + epochs
epochs = [round(i, 2) for i in epochs]
steps = [int(i*step_per_epoch) for i in epochs]
np.random.shuffle(steps)

print(np.array(steps))
print(len(steps))

328.87578125000005
[   624  29598    131   2400  13155   5130   7893   4571   2367   3782
   5558   5360  16443    920   5064   7564  13483   4834   4735   2959
  18417   3387   1118  11181   5459   4012   2828   5097  20390   9208
   4670   6445   4900  78930    328   3190   4933  42753   5886   5689
   9537   6478  31572   1874     98   1578  52620    986   4176   1545
    394  36176  55908  15457   4242   6511   1414  92085   5492   4439
   1644   4406   6051   6018   4966   5755   4341   3617   1052   1743
   6577  10852  85507   5294  15786   5623   5656  11510  25652  17759
   2499   3124  98662   1775   3157  32887   4374   5525   1611   6149
  12497  39465   2532   2894  17101   6182   4538  21048   2598   3222
   1381   2861    295    164   5722   5985   3288   3321 111817   8221
  21705  26310     65   1808  12826 105240   1907   4801    493  22363
   6248   4867  28283 124972  65775   1512   4308  59197   4472   6347
   6281    723  16114   4275   3420   2696   3716      0  

In [5]:
job_count = 0
overwrite = False
print(f" - Number of models to be trained: {len(steps)!r}")

cpu_gpu = "gpu"

for model_str in ["scanvi", "scvi"]:
    
    for epoch in steps:
        
        output_dir_path = os.path.join(dataset_dir, f"model_suo_incremental_training_{model_str}_epoch_{epoch}")
        log_file = os.path.join(logs_directory, f"slurm_out_model_suo_incremental_training_{cpu_gpu}_{model_str}_epoch_{epoch}.log")
        job_name = f"incr_{cpu_gpu}_{model_str}_{epoch}"
        python_name = f"model_training.py"

        if is_job_running(job_name):
            print(f"Training {job_name!r} on {cpu_gpu!r} keeps going for model {model_str!r} and for epoch {epoch!r}.")
        elif overwrite or not os.path.exists(output_dir_path) or not os.path.isdir(output_dir_path):
            try:
                slurm_script = f"""#!/bin/bash
#SBATCH -J {job_name}
#SBATCH -p gpu_p
#SBATCH --qos=gpu_normal
#SBATCH --gres=gpu:1
#SBATCH -c 6
#SBATCH --mem=159G
#SBATCH --nice=0
#SBATCH -t 23:50:00
#SBATCH -o {log_file}
#SBATCH -e {log_file}

source activate sctram_dev_env
python -u {os.path.join(helpers_directory, python_name)} --epoch "{epoch}" --model_str "{model_str}" --overwrite "{overwrite}"
    """
                script_name = os.path.join(logs_directory, f"slurm_job_model_suo_incremental_training_{cpu_gpu}_{model_str}_epoch_{epoch}.sh")
                with open(script_name, "w") as f:
                    f.write(slurm_script)

                print(f"Submitted job on {cpu_gpu!r} {job_count+1} on {cpu_gpu!r}: {job_name!r} for model {model_str!r} and for epoch {epoch!r}")
                subprocess.run(["sbatch", script_name])
                job_count += 1
            finally:
                # time.sleep(0.05)
                os.remove(script_name)

            assert is_job_running(job_name), f"Training {job_name!r} on {cpu_gpu!r} error: model {model_str!r} and for epoch {epoch!r}."
        else:
            print(f"Model exists for model {model_str!r} and for epoch {epoch!r}")
    # if job_count > 32:
    #     break

print(f" - Number of jobs submitted: {job_count}")

 - Number of models to be trained: 276
Submitted job on 'gpu' 1 on 'gpu': 'incr_gpu_scanvi_624' for model 'scanvi' and for epoch 624
Submitted batch job 33823342
Submitted job on 'gpu' 2 on 'gpu': 'incr_gpu_scanvi_29598' for model 'scanvi' and for epoch 29598
Submitted batch job 33823343
Submitted job on 'gpu' 3 on 'gpu': 'incr_gpu_scanvi_131' for model 'scanvi' and for epoch 131
Submitted batch job 33823344
Submitted job on 'gpu' 4 on 'gpu': 'incr_gpu_scanvi_2400' for model 'scanvi' and for epoch 2400
Submitted batch job 33823345
Submitted job on 'gpu' 5 on 'gpu': 'incr_gpu_scanvi_13155' for model 'scanvi' and for epoch 13155
Submitted batch job 33823346
Submitted job on 'gpu' 6 on 'gpu': 'incr_gpu_scanvi_5130' for model 'scanvi' and for epoch 5130
Submitted batch job 33823347
Submitted job on 'gpu' 7 on 'gpu': 'incr_gpu_scanvi_7893' for model 'scanvi' and for epoch 7893
Submitted batch job 33823348
Submitted job on 'gpu' 8 on 'gpu': 'incr_gpu_scanvi_4571' for model 'scanvi' and for e

In [6]:
1

1